# VSAC Value Set Versions by Year

Queries NLM's **FHIR Terminology Service for VSAC** for a list of value set OIDs and
builds a table of published definition versions bucketed into year columns.

**Grouping rule.** By default the year column comes from the leading `YYYY` of each
`version` string (`GROUP_BY = "version"`). Set `GROUP_BY = "date"` to bucket by the
`ValueSet.date` (publication) year instead. If the chosen field can't be parsed for a
given record, the other field is used as a fallback; if neither parses, the version is
reported in the notes cell rather than dropped.

**Retired tag.** Each version carries the FHIR `ValueSet.status` from its own resource.
VSAC collapses Not Maintained / Deprecated / Retired into FHIR `retired`, so any version
that isn't `active` is suffixed `(ret)` in the table. In practice each OID has a single
`active` version (the latest), shown without a suffix.

**Out-of-range years.** Versions earlier than the first year land in a leading catch-all
column (e.g. `≤2022`). Versions later than the last year land in a trailing column
(e.g. `≥2027`) that appears only when such versions exist.

**Auth.** Basic auth with your UMLS API Key: username `apikey`, password = the key.
Set it in your environment before launching Jupyter:

```bash
export UMLS_API_KEY='xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx'
```

**Rate limits.** NLM caps requests at 20/sec per IP and recommends caching results for
12-24h. This notebook throttles requests and writes a local file cache (`.vsac_cache/`);
delete that folder to force a refresh.

In [1]:
import os, time, json, hashlib, pathlib
from collections import defaultdict
import requests
import pandas as pd

# ------------------------- CONFIG -------------------------
UMLS_API_KEY = os.environ.get("UMLS_API_KEY")          # required
BASE = "https://cts.nlm.nih.gov/fhir"                   # version-less base defaults to R4;
                                                       # "https://cts.nlm.nih.gov/fhir/r4" also works
CANONICAL_PREFIX = "http://cts.nlm.nih.gov/fhir/ValueSet/"
YEARS = [2023, 2024, 2025, 2026]                       # explicit columns (range endpoints define span)
GROUP_BY = "version"                                   # "version" -> year from version string prefix
                                                       # "date"    -> year from ValueSet.date
RET_SUFFIX = "(ret)"                                   # appended to any non-active version
REQUESTS_PER_SEC = 10                                  # stay under NLM's 20/sec cap
SLEEP = 1.0 / REQUESTS_PER_SEC
USE_CACHE = True
CACHE_DIR = pathlib.Path(".vsac_cache")

LO, HI = min(YEARS), max(YEARS)
YEAR_COLS = [str(y) for y in range(LO, HI + 1)]        # contiguous, no gaps
EARLIER_LABEL = f"≤{LO - 1}"                       # e.g. "≤2022"
LATER_LABEL   = f"≥{HI + 1}"                       # e.g. "≥2027"

# Paste OIDs or full canonical URLs (either form accepted), one per line.
# To load from a file instead:  OIDS = pathlib.Path("oids.txt").read_text().split()
OIDS = """
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1010.4
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1010.5
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1010.6
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1021.101
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1021.102
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1021.103
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1021.121
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1021.24
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1021.32
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1021.33
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1099.12
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1099.24
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1099.27
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1099.30
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1099.53
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1099.54
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1099.59
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1114.14
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1115.22
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1115.40
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1115.41
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1166.22
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1186.1
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1186.3
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1186.4
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1186.5
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1186.8
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1240.10
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1240.11
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1240.12
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1240.3
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1240.4
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1240.7
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1240.8
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.10
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.11
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.13
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.14
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.15
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.16
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.22
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.23
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.24
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.25
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.29
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.3
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.4
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.5
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.6
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.7
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.8
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113762.1.4.1267.9
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113883.1.11.10267
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113883.1.11.10612
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113883.1.11.14914
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113883.11.20.9.52
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113883.11.20.9.69.4
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113883.3.88.12.3221.8.7
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113883.3.88.12.80.17
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113883.3.88.12.80.62
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113883.4.642.2.575
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113883.4.642.40.2.48.1
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.113883.4.642.40.2.48.3
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.114222.4.11.1066
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.114222.4.11.3591
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.114222.4.11.836
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.114222.4.11.837
http://cts.nlm.nih.gov/fhir/ValueSet/2.16.840.1.114222.4.11.877
""".split()

assert UMLS_API_KEY, "Set the UMLS_API_KEY environment variable before running."
if USE_CACHE:
    CACHE_DIR.mkdir(exist_ok=True)

In [2]:
def to_oid(s):
    s = s.strip()
    return s.rsplit("/", 1)[-1] if s else ""

def to_canonical(oid):
    return CANONICAL_PREFIX + oid

def _cache_path(canonical):
    h = hashlib.sha1(canonical.encode()).hexdigest()[:16]
    return CACHE_DIR / f"{h}.json"

def fetch_versions(canonical, session):
    """All published definition versions for one canonical URL (active + retired buckets),
    deduped by version. Returns list of {version, date, fhir_status}."""
    seen = {}
    for status in ("active", "retired"):            # active first, so it wins on dedup
        url = f"{BASE}/ValueSet"
        params = {"url": canonical, "status": status}
        while url:
            r = session.get(url, params=params, timeout=60)
            params = None                      # 'next' link already carries the query
            if r.status_code == 404:
                break
            r.raise_for_status()
            bundle = r.json()
            for entry in bundle.get("entry", []) or []:
                res = entry.get("resource", {})
                if res.get("resourceType") != "ValueSet":
                    continue
                ver = res.get("version")
                if ver is None:
                    continue
                seen.setdefault(ver, {
                    "version": ver,
                    "date": res.get("date"),
                    "fhir_status": res.get("status"),   # authoritative: 'active' or 'retired'
                })
            url = next((l.get("url") for l in bundle.get("link", []) or []
                        if l.get("relation") == "next"), None)
            time.sleep(SLEEP)
    return list(seen.values())

def fetch_versions_cached(canonical, session):
    if USE_CACHE:
        p = _cache_path(canonical)
        if p.exists():
            return json.loads(p.read_text())
    recs = fetch_versions(canonical, session)
    if USE_CACHE:
        _cache_path(canonical).write_text(json.dumps(recs))
    return recs

def _yr(s):
    s = str(s or "")
    return int(s[:4]) if s[:4].isdigit() else None

def year_of(rec):
    """Year (int) using GROUP_BY field, falling back to the other field; None if neither parses."""
    primary = rec.get("version") if GROUP_BY == "version" else rec.get("date")
    other   = rec.get("date")    if GROUP_BY == "version" else rec.get("version")
    return _yr(primary) if _yr(primary) is not None else _yr(other)

def column_for(year):
    if year < LO:  return EARLIER_LABEL
    if year > HI:  return LATER_LABEL
    return str(year)

def label(rec):
    """Version string, suffixed with RET_SUFFIX when the FHIR status isn't 'active'."""
    v = rec["version"]
    return v if (rec.get("fhir_status") or "").lower() == "active" else v + RET_SUFFIX

In [3]:
session = requests.Session()
session.auth = ("apikey", UMLS_API_KEY)
session.headers.update({"Accept": "application/fhir+json"})

table = {}                       # oid -> {column_label: [display strings]}
detail = []                      # long-format rows
unknown = defaultdict(list)      # oid -> [versions with unparseable year]
errors = {}
any_later = False

oids = [to_oid(x) for x in OIDS if to_oid(x)]
for oid in oids:
    canonical = to_canonical(oid)
    try:
        recs = fetch_versions_cached(canonical, session)
    except requests.HTTPError as ex:
        errors[oid] = str(ex)
        recs = []
    buckets = defaultdict(list)
    # sort oldest-to-newest by version so each cell reads left-to-right chronologically
    for rec in sorted(recs, key=lambda r: str(r.get("version", ""))):
        y = year_of(rec)
        detail.append({"OID": oid, "version": rec["version"], "date": rec.get("date"),
                       "fhir_status": rec.get("fhir_status"), "year": y})
        if y is None:
            unknown[oid].append(rec["version"])
            continue
        col = column_for(y)
        if col == LATER_LABEL:
            any_later = True
        buckets[col].append(label(rec))
    table[oid] = dict(buckets)
    n_ret = sum(1 for r in recs if (r.get("fhir_status") or "").lower() != "active")
    print(f"{oid}: {len(recs)} versions ({len(recs) - n_ret} active, {n_ret} retired), "
          f"{len(unknown[oid])} unparseable")
    time.sleep(SLEEP)

2.16.840.1.113762.1.4.1: 2 versions (1 active, 1 retired), 0 unparseable
2.16.840.1.113762.1.4.1010.4: 2 versions (1 active, 1 retired), 0 unparseable
2.16.840.1.113762.1.4.1010.5: 3 versions (1 active, 2 retired), 0 unparseable
2.16.840.1.113762.1.4.1010.6: 8 versions (1 active, 7 retired), 0 unparseable
2.16.840.1.113762.1.4.1021.101: 4 versions (1 active, 3 retired), 0 unparseable
2.16.840.1.113762.1.4.1021.102: 1 versions (1 active, 0 retired), 0 unparseable
2.16.840.1.113762.1.4.1021.103: 1 versions (1 active, 0 retired), 0 unparseable
2.16.840.1.113762.1.4.1021.121: 2 versions (1 active, 1 retired), 0 unparseable
2.16.840.1.113762.1.4.1021.24: 3 versions (1 active, 2 retired), 0 unparseable
2.16.840.1.113762.1.4.1021.32: 3 versions (1 active, 2 retired), 0 unparseable
2.16.840.1.113762.1.4.1021.33: 1 versions (1 active, 0 retired), 0 unparseable
2.16.840.1.113762.1.4.1099.12: 2 versions (1 active, 1 retired), 0 unparseable
2.16.840.1.113762.1.4.1099.24: 3 versions (1 active, 2 re

In [4]:
columns = ["OID", EARLIER_LABEL] + YEAR_COLS + ([LATER_LABEL] if any_later else [])

rows = []
for oid in oids:
    row = {"OID": oid}
    for c in columns[1:]:
        row[c] = ",".join(table.get(oid, {}).get(c, []))
    rows.append(row)

df = pd.DataFrame(rows, columns=columns)
df

,OID,≤2022,2023,2024,2025,2026
0,2.16.840.1.113762.1.4.1,"20121025(ret),20150331",,,,
1,2.16.840.1.113762.1.4.1010.4,20170601(ret),,20240606,,
2,2.16.840.1.113762.1.4.1010.5,20190423(ret),,"20240606(ret),20240627",,
3,2.16.840.1.113762.1.4.1010.6,"20160903(ret),20180107(ret),20180519(ret),2021...",20230715(ret),,20250411(ret),20260428
4,2.16.840.1.113762.1.4.1021.101,"20211028(ret),20220316(ret)",,20240215(ret),,20260429
...,...,...,...,...,...,...
64,2.16.840.1.114222.4.11.1066,20190521(ret),,20240606,,
65,2.16.840.1.114222.4.11.3591,20221118(ret),,,20250419,
66,2.16.840.1.114222.4.11.836,20121025,,,,
67,2.16.840.1.114222.4.11.837,20121025,,,,


In [7]:
def to_md(frame):
    """Markdown table with a 1-based '#' column taken from the DataFrame index."""
    cols = list(frame.columns)
    head = ["#"] + cols
    out = ["| " + " | ".join(head) + " |",
           "| " + " | ".join(["---"] * len(head)) + " |"]
    for idx, r in frame.iterrows():
        out.append("| " + " | ".join([str(idx + 1)] + [str(r[c]) for c in cols]) + " |")
    return "\n".join(out)

md = to_md(df)
print(md)

pathlib.Path("vsac_versions_by_year.md").write_text(md + "\n")
df.to_csv("vsac_versions_by_year.csv", index=False)
pd.DataFrame(detail).to_csv("vsac_versions_detail.csv", index=False)
print("\nWrote: vsac_versions_by_year.md, vsac_versions_by_year.csv, vsac_versions_detail.csv")

| # | OID | ≤2022 | 2023 | 2024 | 2025 | 2026 |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 2.16.840.1.113762.1.4.1 | 20121025(ret),20150331 |  |  |  |  |
| 2 | 2.16.840.1.113762.1.4.1010.4 | 20170601(ret) |  | 20240606 |  |  |
| 3 | 2.16.840.1.113762.1.4.1010.5 | 20190423(ret) |  | 20240606(ret),20240627 |  |  |
| 4 | 2.16.840.1.113762.1.4.1010.6 | 20160903(ret),20180107(ret),20180519(ret),20210630(ret),20220701(ret) | 20230715(ret) |  | 20250411(ret) | 20260428 |
| 5 | 2.16.840.1.113762.1.4.1021.101 | 20211028(ret),20220316(ret) |  | 20240215(ret) |  | 20260429 |
| 6 | 2.16.840.1.113762.1.4.1021.102 | 20211028 |  |  |  |  |
| 7 | 2.16.840.1.113762.1.4.1021.103 | 20211113 |  |  |  |  |
| 8 | 2.16.840.1.113762.1.4.1021.121 |  |  | 20241007(ret) | 20250228 |  |
| 9 | 2.16.840.1.113762.1.4.1021.24 | 20190308(ret),20211119(ret) |  | 20240330 |  |  |
| 10 | 2.16.840.1.113762.1.4.1021.32 | 20211028(ret),20220316(ret) |  | 20240215 |  |  |
| 11 | 2.16.840.1.113762.1.4.1021.33 | 2021112

In [6]:
# Records with an unparseable year, plus any request errors
if any(unknown.values()):
    print("Versions with no parseable year (not placed in any column):")
    for oid, vs in unknown.items():
        if vs:
            print("  " + oid + ": " + ", ".join(vs))
if errors:
    print("\nErrors:")
    for oid, e in errors.items():
        print("  " + oid + ": " + e)
if not any(unknown.values()) and not errors:
    print("All versions placed; no errors.")

All versions placed; no errors.
